# Car Price Prediction — Data Loading

## Project Overview

This project predicts used car prices in Pakistan using real listings
scraped from PakWheels — the largest used car marketplace in Pakistan.

The dataset contains 18 columns including price, mileage, engine size,
transmission, fuel type, city, and seller information.

## This Notebook Covers

- Loading the raw scraped data
- Inspecting all 18 columns and their data types
- Identifying dirty columns that need cleaning
- Checking missing values
- Saving a raw snapshot for reproducibility

## Problems We Will Solve in This Project

| Problem | Description |
|---------|-------------|
| Dirty price column | Stored as "PKR 38.75 lacs" — needs to become a number |
| Dirty mileage column | Stored as "19,198 km" — needs commas and text removed |
| Dirty engine column | Stored as "660 cc" — needs unit removed |
| Empty make column | Brand name needs to be extracted from the title |
| Missing values | Several columns have empty cells that need handling |
| Unrealistic years | Some year values may be invalid and need filtering |
| Text categories | Transmission, fuel type need encoding for ML model |

## 1. Importing Libraries

We need two libraries for this notebook.
Pandas loads and works with the data.
OS builds file paths that work on any computer.

In [1]:
import pandas as pd
import os

No output means both libraries loaded without any errors.
Every notebook in this project starts with these two imports.

## 2. Loading the Raw Data

The dataset is a CSV file scraped directly from PakWheels.
It lives in the data/raw folder.
We load it exactly as it is — no changes yet.

In [2]:
data_path = os.path.join('..', 'data', 'raw', 'pakwheels.csv')
df = pd.read_csv(data_path)
print(f"Rows loaded: {len(df)}")
print(f"Columns: {df.shape[1]}")

Rows loaded: 1394
Columns: 18


The numbers above tell us how many car listings we have
and how many columns of information exist for each listing.

Each row is one used car listed for sale on PakWheels.
Each column tells us something about that car.

## 3. First Look at the Data

Before doing anything, we peek at the first few rows.
This gives us a feel for what the data actually looks like
before we start analyzing or cleaning it.

In [3]:
df.head(3)

,title,url,price,year,mileage,engine,transmission,fuel_type,city,seller_type,ad_id,posted_date,scraped_at,source_type,source_url,make,model_slug,record_type
0,Honda N Box 2022 Custom GL for Sale9.7/10,https://www.pakwheels.com/used-cars/honda-n-bo...,PKR 38.75 lacs,2022.0,"19,198 km",660 cc,Automatic,Petrol,Lahore,Individual,main_ad_10843516,Automatic,4/23/2026 0:00,used-car,https://www.pakwheels.com/used-cars/search/-/,NaN,NaN,NaN
1,Honda N Wgn 2023 L for Sale9.5/10,https://www.pakwheels.com/used-cars/honda-n-wg...,PKR 40.25 lacs,2023.0,"20,997 km",660 cc,Automatic,Petrol,Lahore,Individual,main_ad_10865190,Automatic,4/23/2026 0:00,used-car,https://www.pakwheels.com/used-cars/search/-/,NaN,NaN,NaN
2,Toyota Hilux 2004 Double Cab for Sale,https://www.pakwheels.com/used-cars/toyota-hil...,PKR 35 lacs,2004.0,"180,000 km",3000 cc,Manual,Diesel,Peshawar,Individual,main_ad_11347129,Manual,4/23/2026 0:00,used-car,https://www.pakwheels.com/used-cars/search/-/,NaN,NaN,NaN


Even from just 3 rows we can already spot problems.

Price is stored as text like "PKR 38.75 lacs" instead of a number.
Mileage is stored as "19,198 km" instead of a number.
Engine is stored as "660 cc" instead of a number.
The make and model_slug columns appear to be empty.

We will fix all of these in the cleaning notebook.
For now we just observe and document.

## 4. Column Names and Data Types

Now we look at all 18 columns at once.
This tells us the name of each column and what type of data it holds.
It also shows us which columns have missing values.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1394 entries, 0 to 1393
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         1331 non-null   object 
 1   url           1331 non-null   object 
 2   price         1331 non-null   object 
 3   year          1331 non-null   float64
 4   mileage       1331 non-null   object 
 5   engine        605 non-null    object 
 6   transmission  605 non-null    object 
 7   fuel_type     1331 non-null   object 
 8   city          1331 non-null   object 
 9   seller_type   1331 non-null   object 
 10  ad_id         1331 non-null   object 
 11  posted_date   1331 non-null   object 
 12  scraped_at    1331 non-null   object 
 13  source_type   1331 non-null   object 
 14  source_url    1331 non-null   object 
 15  make          0 non-null      float64
 16  model_slug    0 non-null      float64
 17  record_type   0 non-null      float64
dtypes: float64(4), object(14)
me

A few things to note from the output above.

Most columns are stored as object which means Python sees them as text.
Price, mileage, and engine need to be converted to numbers.
InvoiceDate equivalent here is posted_date which also needs fixing.

Any column where the non-null count is less than the total row count
has missing values that we need to handle in the next notebook.

## 5. Checking Missing Values

Now we count exactly how many values are missing in each column.
A missing value means that piece of information was not available
for that particular car listing on PakWheels.

In [5]:
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

record_type     1394
model_slug      1394
make            1394
engine           789
transmission     789
ad_id             63
source_url        63
source_type       63
scraped_at        63
posted_date       63
title             63
url               63
city              63
fuel_type         63
mileage           63
year              63
price             63
seller_type       63
dtype: int64


The columns listed above have missing data.
The number next to each column is how many rows are missing that value.

Columns like make and model_slug are likely empty for most rows
because the scraper did not extract them — we will build those ourselves
from the title column in the cleaning notebook.

## 6. Inspecting the Dirty Columns

We already saw that price, mileage, and engine are stored as text.
Let us look at the unique formats in each of these columns
so we know exactly what we are dealing with before cleaning.

In [6]:
print("Price samples:")
print(df['price'].dropna().head(5).values)
print("\nMileage samples:")
print(df['mileage'].dropna().head(5).values)
print("\nEngine samples:")
print(df['engine'].dropna().head(5).values)

Price samples:
['PKR 38.75 lacs' 'PKR 40.25 lacs' 'PKR 35 lacs' 'PKR 88.5 lacs'
 'PKR 56.85 lacs']

Mileage samples:
['19,198 km' '20,997 km' '180,000 km' '97,000 km' '82,000 km']

Engine samples:
['660 cc' '660 cc' '3000 cc' '3000 cc' '1800 cc']


Now we can see exactly what needs to be fixed.

Price has "PKR" and "lacs" attached — we need to extract just the number
and convert lacs to actual rupees (1 lac = 100,000).

Mileage has commas and "km" attached — we need to remove those
and convert to a plain integer.

Engine has "cc" attached — we need to remove it and convert to a number.

All three of these will be fixed in the cleaning notebook.

## 7. Checking Key Categorical Columns

Let us look at the unique values in the categorical columns.
This tells us how many cities, transmission types, and fuel types exist
and whether there are any unexpected or misspelled values.

In [7]:
print(f"Cities: {df['city'].nunique()}")
print(f"Transmission: {df['transmission'].unique()}")
print(f"Fuel types: {df['fuel_type'].unique()}")
print(f"Seller types: {df['seller_type'].unique()}")

Cities: 87
Transmission: ['Automatic' 'Manual' nan]
Fuel types: ['Petrol' 'Diesel' 'Hybrid' 'Electric' 'LPG' 'PHEV' 'CNG' '4 Stroke'
 '2 Stroke' nan]
Seller types: ['Individual' nan]


This gives us a clear picture of the categorical data.

The number of unique cities tells us how spread out the listings are
across Pakistan — more cities means more diverse market data.

Transmission, fuel type, and seller type should each have
a small number of categories. Any unexpected values here
would be data entry errors that need fixing in the next notebook.

## 8. Checking the Year Column

The year column tells us what model year each car is.
We want to know the range — oldest to newest car in the dataset.
We also want to check if there are any unrealistic year values.

In [8]:
print(f"Oldest car: {df['year'].min()}")
print(f"Newest car: {df['year'].max()}")
print(f"Missing years: {df['year'].isnull().sum()}")

Oldest car: 1978.0
Newest car: 2026.0
Missing years: 63


The year range tells us how old the oldest car in the dataset is.
Any year before 1980 or after 2026 would be a data error.
We will filter out unrealistic years in the cleaning notebook.

The car age will also become an important feature for our model later —
older cars generally sell for less than newer ones.

## 9. Top Car Makes in the Dataset

The title column contains the full listing name like "Honda Civic 2020".
The make column is empty so we need to see which brands appear most
by looking at the title column directly.

In [9]:
top_makes = df['title'].str.split().str[0].value_counts().head(10)
print(top_makes)

title
Honda       429
Suzuki      203
Toyota      194
Yamaha       55
Super        39
Daihatsu     32
Hi           27
United       23
Hyundai      23
KIA          22
Name: count, dtype: int64


These are the top 10 car brands by number of listings in the dataset.

Toyota and Honda are likely at the top because they are the most
popular brands in the Pakistani used car market.

This quick extraction from the title column confirms that we can
build the make column ourselves in the cleaning notebook.

## 10. Saving a Raw Snapshot

Before we do anything else, we save an exact copy of the raw data.
This is important for reproducibility — if anything goes wrong later,
we can always come back to this exact starting point.

In [10]:
snapshot_path = os.path.join('..', 'data', 'processed', '01_raw_snapshot.csv')
df.to_csv(snapshot_path, index=False)
print(f"Raw snapshot saved: {len(df)} rows, {df.shape[1]} columns")

Raw snapshot saved: 1394 rows, 18 columns


## 11. Data Loading Summary

Here is everything we found in this notebook.
This is our starting point before any cleaning happens.

In [11]:
print(f"Total listings:       {len(df)}")
print(f"Total columns:        {df.shape[1]}")
print(f"Columns with nulls:   {(df.isnull().sum() > 0).sum()}")
print(f"Year range:           {int(df['year'].min())} - {int(df['year'].max())}")
print(f"Unique cities:        {df['city'].nunique()}")
print(f"Top brand:            {df['title'].str.split().str[0].value_counts().index[0]}")

Total listings:       1394
Total columns:        18
Columns with nulls:   18
Year range:           1978 - 2026
Unique cities:        87
Top brand:            Honda


This summary captures the state of the raw data in one glance.

The next notebook, 02_data_cleaning, will fix every problem we found here.

---

## Quick Reference — Methods Used in This Notebook

| Code | What it does |
|------|-------------|
| `pd.read_csv(path)` | Loads a CSV file into a DataFrame |
| `df.shape` | Returns (rows, columns) as a tuple |
| `df.head(3)` | Shows the first 3 rows |
| `df.info()` | Shows column names, types, and null counts |
| `df.isnull().sum()` | Counts missing values per column |
| `df['col'].nunique()` | Counts how many unique values are in a column |
| `df['col'].unique()` | Shows all unique values in a column |
| `df['col'].min()` | Returns the smallest value |
| `df['col'].max()` | Returns the largest value |
| `str.split().str[0]` | Splits text and takes the first word |
| `value_counts().head(10)` | Shows top 10 most frequent values |
| `df.to_csv(path, index=False)` | Saves DataFrame to CSV without row numbers |
| `os.path.join(...)` | Builds a file path that works on any OS |